In [112]:
import pandas as pd
import numpy as np
from collections import Counter

##### Assumptions
- 1 year of a tender is allocated 1 year of antigen demand (through 1 or many vaccines); scales appropriately
- Intenvory is based on demand for a given year.
- Ratio of inventory to supply is calculated based on end of year totals (after calcs), but only in the programatic sense (check after math, add inventory at end of year for next year)

##### Load and setup demand

In [113]:
# Load the CSV file
file_path = 'data/real/antigen_demand_80_20_2_scenarios.csv'
data = pd.read_csv(file_path)
# Create the two dataframes based on the 'prob' column
demand_80 = data[data['prob'] == 0.8]
demand_20 = data[data['prob'] == 0.2]

demand_80 = demand_80.drop(columns=['prob', 'demand_SID'])
demand_20 = demand_20.drop(columns=['prob', 'demand_SID'])
# Expanding the 'demands' column into 10 separate columns
demand_80_expanded = demand_80['demands'].apply(lambda x: pd.Series(eval(x)))
demand_20_expanded = demand_20['demands'].apply(lambda x: pd.Series(eval(x)))

# Renaming the columns to 1-10
demand_80_expanded.columns = range(1, 11)
demand_20_expanded.columns = range(1, 11)

# Concatenating the expanded demands columns back to the original antigen column
demand_80_final = pd.concat([demand_80['antigen'], demand_80_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_80_final[11] = 0.1
demand_20_final = pd.concat([demand_20['antigen'], demand_20_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_20_final[11] = 0.1

# demand_80_final.head(), demand_20_final.head()


##### Load and setup Starting Points

In [114]:
file_path = 'data/real/Starting_point.xlsx'

# Load the sheets 'F_start', 'I_start', 'S_start' into their own DataFrames
f_start = pd.read_excel(file_path, sheet_name='F_start')
i_start = pd.read_excel(file_path, sheet_name='I_start')
s_start = pd.read_excel(file_path, sheet_name='S_start')

##### Initialize stuff

In [115]:
#tender length
delta = 3
vaccine_consumption_percent = 1
years = 10

# Creating an empty DataFrame with the specified structure for calculating ratios
antigens = f_start['Antigen']
columns = ['Antigen',1]

ratio_DF = pd.DataFrame(columns=columns)
ratio_DF['Antigen'] = antigens
ratio_DF[1] = np.zeros(len(antigens))

# create DF to store tender schedule
tender_schedules = f_start.copy()

########################################################
#create DF to store current inventory
inventory_DF = i_start.copy()
##########################################################
#create DF to store unvaccinated children


In [116]:
import pandas as pd

def calculate_coverage_and_ratios(inventory_DF, demand_DF, V_a, year):
    # Initialize the total coverage dictionary
    total_coverage = {antigen: 0 for antigen in V_a.keys()}

    # Iterate across each row in the inventory DataFrame
    for index, row in inventory_DF.iterrows():
        vaccine = row['Vaccine']
        amount = row['Amount']
        
        # For each antigen covered by the vaccine, add the amount to the coverage
        for antigen in V_a.keys():
            if vaccine in V_a[antigen]:
                total_coverage[antigen] += amount

    # Convert the total coverage dictionary to a DataFrame
    total_coverage_df = pd.DataFrame(list(total_coverage.items()), columns=['antigen', 'Total_Coverage'])

    # Calculate ratio of supply and demand
    calculate_ratios_DF = pd.merge(demand_DF.iloc[:, [0, year]], total_coverage_df, left_on='antigen', right_on='antigen')
    calculate_ratios_DF['Ratio'] = calculate_ratios_DF['Total_Coverage'] / calculate_ratios_DF[year]

    # print(f"Ratio DF:\n{calculate_ratios_DF[['antigen', 'Ratio']]}")

    return calculate_ratios_DF

# Example usage:
# result_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year)


##### Initialize antigens/vaccines/prodicers

In [102]:
A = ["Measles", "Mumps", "Rubella"]
V = ["M", "MR", "MMR"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"]
}

P = ["Biological_E", 
    "GSK","PT_Bio", 
    "Serum_Institute"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}

#translate vaccine - antigen, to antigen - vaccine
V_a = {}

for a in A:
    vector_a = []
    for v in A_v.keys():
        if a in A_v[v]:
            vector_a.append(v)
    V_a[a] = vector_a

V_a



{'Measles': ['M', 'MR', 'MMR'], 'Mumps': ['MMR'], 'Rubella': ['MR', 'MMR']}

## TESTING - Measles Containing Vaccines Only

In [103]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
demand_80_MCV = demand_80_final[demand_80_final['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
f_start_MCV = f_start[f_start['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
# i_start_MCV = i_start[i_start['Vaccine'].isin(['M', 'MR', 'MMR'])]
s_start_MCV = s_start[s_start['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
ratio_DF_MCV = pd.DataFrame()
inventory_DF_MCV = inventory_DF[inventory_DF['Vaccine'].isin(['M', 'MR', 'MMR'])]


##### Logic to translate vaccine totals to antigen coverage for later math

In [104]:
# Initialize the total coverage dictionary
total_coverage = {antigen: 0 for antigen in V_a.keys()}

# Iterate across each row in the DataFrame
for index, row in inventory_DF_MCV.iterrows():
    vaccine = row['Vaccine']
    amount = row['Amount']
    
    # For each antigen covered by the vaccine, add the amount to the coverage
    for antigen in V_a.keys():
        if vaccine in V_a[antigen]:
            total_coverage[antigen] += amount

# Convert the total coverage dictionary to a DataFrame
total_coverage_df = pd.DataFrame(list(total_coverage.items()), columns=['antigen', 'Total_Coverage'])

# Output the total coverage for each antigen
total_coverage_df


,antigen,Total_Coverage
0,Measles,1.173546e+09
1,Mumps,7.846400e+07
2,Rubella,9.159641e+08


In [105]:
#logic to setup least covered antigens:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)

In [106]:
demand_80_MCV

,antigen,1,2,3,4,5,6,7,8,9,10,11
10,Measles,420097000,358616900,394831600,346093200,377569100,407486200,271512200,386591100,290515900,326808200,0.1
12,Mumps,26052200,26626400,25785400,25268100,23536200,16205400,18650400,8909600,10718500,12977700,0.1
20,Rubella,357331300,254011400,304621400,276893800,291347300,386542700,259074600,385691400,289592400,325856600,0.1


In [117]:
for year in range(1, 4 + 1):  # Iterate through each year - short range for testing  range(1,len(demand_80_MCV.columns)-1)
    print("********************HAPPY NEW YEAR****************************")
    print("******************RETICULATING SPLINE**************************")
    print(f"Year: {year}")

    least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)
    remaining_inventory = {}
    uncovered_demand = {}
    while least_covered_antigens:  # Iterate through each antigen, find what vaccines cover each antigen, least to greatest, update supply and demand
        print('###############################################################')
        antigen = least_covered_antigens.pop(0)
        print(f"serving antigen {antigen}")
        for vaccine, antigens in A_v.items():  # Iterate through A_v to check which vaccines cover the antigen
            if antigen in antigens and demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year] >0:

                vaccine_inventory_value = inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine].iloc[0, 1]
                # print("iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii")
                # print(f"Inventory for {vaccine} for year {year}: ", vaccine_inventory_value)

                antigen_demand_value = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year]
                # print(f"Demand for {antigen} for year {year}: ", antigen_demand_value)

                difference = vaccine_inventory_value - antigen_demand_value
                if difference >= 0: #
                    remaining_inventory[vaccine] = difference
                    decrement = antigen_demand_value
                else: 
                    remaining_inventory[vaccine] = 0
                    decrement = vaccine_inventory_value
                    uncovered_demand[antigen] = abs(difference)
                    print("------------------------------------------")
                    # print(f"Vaccine:3 {vaccine}, antigen: {antigen}")
                    print(f"uncovered demand for {antigen}: {[antigen]}")
                    #transfer uncovered demand to next year

                # print("^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
                inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine, inventory_DF_MCV.columns[1]] = remaining_inventory[vaccine]

                print("Decrementing antigen demands")
                for ant in antigens:
                    if demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]].item() > 0:
                        # print(f"from {ant} demand, reducing demand for year {year} for antigen {ant} by {decrement}")
                        demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]] = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant].iloc[0, year] - decrement

        print(f"{len(antigen_counts)} antigens entered, only {least_covered_antigens} remain!")

    #update any uncovered demand, to next year. add uncovered demand to dosses_missed dict
    if 'uncovered_demand' in locals(): # Check if the variable exists
        while uncovered_demand:
            top = uncovered_demand.popitem()
            top_antigen = top[0]
            doses_missed = top[1]
            print(f"{doses_missed} doses missed for {top_antigen}")
            demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == top_antigen, demand_80_MCV.columns[year+1]] += doses_missed
            s_start_MCV.loc[s_start_MCV.iloc[:, 0] == top_antigen, s_start_MCV.columns[1]] += doses_missed
    else:
        print("no uncovered demand this year")
    
    #check ratio for supply/demand.
    ratio_DF_MCV = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year)

    print("Checking ratio of supply to demand for antigens!")
    print()

********************HAPPY NEW YEAR****************************
******************RETICULATING SPLINE**************************
Year: 1
###############################################################
serving antigen Mumps
3 antigens entered, only ['Rubella', 'Measles'] remain!
###############################################################
serving antigen Rubella
3 antigens entered, only ['Measles'] remain!
###############################################################
serving antigen Measles
3 antigens entered, only [] remain!
Checking ratio of supply to demand for antigens!

********************HAPPY NEW YEAR****************************
******************RETICULATING SPLINE**************************
Year: 2
###############################################################
serving antigen Mumps
3 antigens entered, only ['Rubella', 'Measles'] remain!
###############################################################
serving antigen Rubella
3 antigens entered, only ['Measles'] remain!
######

In [128]:
ratio_DF_MCV = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, 6)

# Assuming ratio_DF_MCV is already defined using the function
for antigen in least_covered_antigens:
    print(antigen)
    # Select the Ratio value for the specific antigen
    try:
        ratio_value = ratio_DF_MCV.loc[ratio_DF_MCV['antigen'] == antigen, 'Ratio'].item()
        print(f"Ratio for {antigen}: {ratio_value}")
    except ValueError:
        print(f"No ratio found for {antigen}")


Mumps
Ratio for Mumps: 0.0
Rubella
Ratio for Rubella: 0.0
Measles
Ratio for Measles: 0.0
